In [16]:
# ============================================================
# 필요한 라이브러리 불러오기
# ============================================================

# pandas: 엑셀처럼 표(DataFrame) 형태로 데이터를 다루는 도구입니다.
# 줄여서 pd라고 별명을 붙여줍니다. (관례)
import pandas as pd

# numpy: 수학 연산(평균, 표준편차 등)을 빠르게 해주는 도구입니다.
# 줄여서 np라고 별명을 붙여줍니다. (관례)
import numpy as np

# matplotlib.pyplot: 그래프(차트)를 그려주는 시각화 도구입니다.
# 줄여서 plt라고 별명을 붙여줍니다. (관례)
import matplotlib.pyplot as plt

# seaborn: matplotlib 위에 만들어진 더 예쁜 시각화 도구입니다.
# 줄여서 sns라고 별명을 붙여줍니다. (관례)
import seaborn as sns

# warnings: 파이썬이 뱉는 경고 메시지를 숨겨줍니다.
# (실습에 불필요한 경고가 뜨는 걸 방지하기 위해 사용합니다)
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

print("라이브러리 로드 완료!")

라이브러리 로드 완료!


In [6]:
# ============================================================
# 과제 데이터 로드: Heart Disease (심장병 예측)
# ============================================================

# 이 데이터셋은 환자의 나이, 혈압, 콜레스테롤 등의 정보로
# 심장병 여부(target: 0=정상, 1=심장병)를 예측하는 데이터입니다.

df_mission = pd.read_csv('https://raw.githubusercontent.com/amankharwal/Website-data/master/heart.csv')

print(f"데이터 크기: {df_mission.shape}")
print(f"→ 총 {df_mission.shape[0]}명의 환자, {df_mission.shape[1]}개의 특성")
print()
df_mission.head()

데이터 크기: (303, 14)
→ 총 303명의 환자, 14개의 특성



,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [7]:

# 1단계: EDA (데이터 탐색)
# → df_mission.info(), df_mission.describe(), df_mission.isnull().sum() 등
print("1단계 데이터 탐색")
print(df_mission.info()) # 타입과 데이터 결측치 확인
print("\n2. 통계 요약")
print(df_mission.describe()) # 수치 데이터 분포 확인
print("\n3. 결측치 개수 확인")
print(df_mission.isnull().sum())

1단계 데이터 탐색
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        303 non-null    int64  
 12  thal      303 non-null    int64  
 13  target    303 non-null    int64  
dtypes: float64(1), int64(13)
memory usage: 33.3 KB
None

2. 통계 요약
              age         sex          cp    trestbps        chol         fbs  \
count  303.000000  303.000000  303.000000  303.000000  303.000000  303.000000   
mean    54.366337    0.683168  

In [9]:
# 2단계: 결측치 처리
# → 결측치가 있다면 적절한 방법으로 처리하세요!
print("2단계 결측치 처리")

if df_mission.isnull().sum().sum() > 0:
    df_mission = df_mission.fillna(df_mission.median()) # 결측치가 있는 경우 중앙값으로 채움

2단계 결측치 처리


In [10]:
# 3단계: 이상치 처리 (선택사항)
# → 박스플롯으로 확인하고, 필요하면 IQR 방법으로 제거하세요!
print("3단계 이상치 처리")

Q1 = df_mission['chol'].quantile(0.25)
Q3 = df_mission['chol'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

3단계 이상치 처리


In [11]:
# 4단계: 인코딩
# → 문자열 컬럼을 찾아서 숫자로 바꿔주세요!
# → 힌트: df_mission.select_dtypes(include='object').columns
print("\n4단계 인코딩")

char_cols = df_mission.select_dtypes(include='object').columns
if len(char_cols) > 0:
    df_mission = pd.get_dummies(df_mission, columns=char_cols, drop_first=True)
    print(f"인코딩된 컬럼: {list(char_cols)}")


4단계 인코딩


In [12]:
# 5단계: X, y 분리
# → target 컬럼이 정답(y)입니다!
print("\n5단계 X, y 분리")

X = df_mission.drop(columns=['target']) # target(심장병 여부)을 제외한 모든 것
y = df_mission['target']               # 정답 데이터


5단계 X, y 분리


In [17]:
# 6단계: Train/Test Split
print("\n6단계 Train/Test Split")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


6단계 Train/Test Split


In [18]:
# 7단계: 스케일링
# → fit_transform은 train에만! transform은 test에만!
print("\n7단계 스케일링")

scaler = StandardScaler()
# 학습 데이터는 fit_transform, 테스트 데이터는 transform만! (데이터 누수 떄매)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


7단계 스케일링


In [19]:
# 8단계: 모델 학습 & 정확도 출력
print("\n8단계 모델 학습 & 정확도 출력")

model_mission = RandomForestClassifier(n_estimators=100, random_state=42)
model_mission.fit(X_train_scaled, y_train)

# 예측 및 결과 출력
y_pred = model_mission.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred) * 100

print("-" * 30)
print(f"심장병 예측 모델 정확도: {accuracy:.2f}%")
print("-" * 30)
print(classification_report(y_test, y_pred))


8단계 모델 학습 & 정확도 출력
------------------------------
심장병 예측 모델 정확도: 83.61%
------------------------------
              precision    recall  f1-score   support

           0       0.83      0.83      0.83        29
           1       0.84      0.84      0.84        32

    accuracy                           0.84        61
   macro avg       0.84      0.84      0.84        61
weighted avg       0.84      0.84      0.84        61

